In [1]:
import os
import json
import csv
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, Optional, Literal, Any

import numpy as np
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm

import datasets
import unet
import prof_unet
from SplitNet import SplitNet


# -----------------------------------------------------------------------------
# Global setup
# -----------------------------------------------------------------------------

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------------------------------------------------------
# Type aliases
# -----------------------------------------------------------------------------

ModelType = Literal[
    "splitnet_attn",
    "splitnet",
    "unet",
    "attn_unet",
    "prof_unet",
]

DatasetMode = Literal[
    "border",
    "border_pressure",
    "fixed",
]

TrainingMode = Literal[
    "physics_limited",
    "baseline_full",
]


# -----------------------------------------------------------------------------
# Experiment configuration
# -----------------------------------------------------------------------------

@dataclass
class ExperimentConfig:
    """
    Stores all settings for a single model training run.

    training_mode:
        physics_limited:
            Uses a limited dataset.
            Supervised MSE is only computed at observed/masked points.
            Darcy loss is computed on the full predicted output.

        baseline_full:
            Uses a full-supervision dataset.
            MSE is computed over the whole image.
            Darcy weight must be 0.
    """

    model_type: ModelType = "splitnet_attn"
    dataset_mode: DatasetMode = "fixed"
    training_mode: TrainingMode = "physics_limited"

    mse_weight: float = 1.0
    darcy_weight: float = 1.0

    epochs: int = 250
    batch_size: int = 8
    lr: float = 1e-3

    channels: str = "KP"

    train_sims_path: str = "../train_sims.npy"
    val_sims_path: str = "../val_sims.npy"
    sim_max_exclusive: Optional[int] = 500

    save_dir: str = "results"
    run_name: Optional[str] = None
    save_prefix: Optional[str] = None

    save_best: bool = True
    save_final: bool = True

    # "old_zeroed" matches your older training code.
    # "true_masked" divides only by observed pixels.
    mask_loss_style: str = "old_zeroed"

    # If None:
    #   physics_limited -> val_supervised_mse
    #   baseline_full    -> val_total_mse
    best_metric: Optional[str] = None

    # Optional dataset-specific settings.
    # Example:
    # {"points_per_side": 5, "radius": 3, "steps": (0, 200)}
    dataset_kwargs: Optional[Dict[str, Any]] = None


# -----------------------------------------------------------------------------
# Darcy physics loss
# -----------------------------------------------------------------------------

def darcy_loss_from_output(out: torch.Tensor) -> torch.Tensor:
    """
    Computes mean squared Darcy residual:

        div(K * grad(P))^2

    Expected channel order:
        channel 0 = K
        channel 1 = P

    Input shape:
        [B, C, H, W]

    Returns:
        scalar tensor
    """

    if out.shape[1] < 2:
        raise ValueError("Darcy loss requires at least two channels: K and P.")

    k = out[:, 0:1]
    p = out[:, 1:2]

    p_y, p_x = torch.gradient(p, dim=(-2, -1))

    flux_y = k * p_y
    flux_x = k * p_x

    div_y = torch.gradient(flux_y, spacing=(1,), dim=(-2,))[0]
    div_x = torch.gradient(flux_x, spacing=(1,), dim=(-1,))[0]

    residual = div_y + div_x
    return (residual ** 2).mean()


# -----------------------------------------------------------------------------
# Model factory
# -----------------------------------------------------------------------------

def get_num_channels(channels: str) -> int:
    if channels == "all":
        return 3
    if channels == "KP":
        return 2
    if channels in ["K", "P", "phi"]:
        return 1

    raise ValueError("channels must be 'all', 'KP', 'K', 'P', or 'phi'.")


def make_model(model_type: ModelType, channels: str = "KP") -> nn.Module:
    """
    Creates the requested model architecture.

    SplitNet models are only designed for KP input/output.
    U-Net models can work with KP, all, K, P, or phi.
    """

    model_type = model_type.lower()
    num_channels = get_num_channels(channels)

    if model_type == "splitnet_attn":
        if channels != "KP":
            raise ValueError("SplitNet only supports channels='KP'.")
        return SplitNet(attn=True).to(DEVICE)

    if model_type == "splitnet":
        if channels != "KP":
            raise ValueError("SplitNet only supports channels='KP'.")
        return SplitNet(attn=False).to(DEVICE)

    if model_type == "unet":
        return unet.SmallUnet(channels=num_channels).to(DEVICE)

    if model_type == "attn_unet":
        return unet.AttnUnet(channels=num_channels).to(DEVICE)

    if model_type == "prof_unet":
        # Professor U-Net expects power-of-two-friendly sizes.
        # ResizedUNet internally resizes 200x200 -> 256x256 -> 200x200.
        return prof_unet.ResizedUNet(
            in_channels=num_channels,
            num_classes=num_channels,
            size=256,
        ).to(DEVICE)

    raise ValueError(f"Unknown model_type: {model_type}")


# -----------------------------------------------------------------------------
# Dataset selection
# -----------------------------------------------------------------------------

def get_dataset_class(dataset_mode: DatasetMode, training_mode: TrainingMode):
    """
    Selects the correct dataset class.

    Dense vs Thin:
        Dense = iterates through all steps.
        Thin  = samples fewer/random steps.

    Limited vs Full:
        Limited = returns a mask and trains only on observed points.
        Full    = target is the full image.
    """

    if training_mode == "physics_limited":
        if dataset_mode == "border":
            return datasets.BorderDenseDatasetLimited
        if dataset_mode == "border_pressure":
            return datasets.BorderDensePressureGradientDatasetLimited
        if dataset_mode == "fixed":
            return datasets.FixedDenseDatasetLimited

    if training_mode == "baseline_full":
        if dataset_mode == "border":
            return datasets.BorderDenseDatasetFull
        if dataset_mode == "border_pressure":
            return datasets.BorderDensePressureGradientDatasetFull
        if dataset_mode == "fixed":
            return datasets.FixedDenseDatasetFull

    raise ValueError(
        f"Invalid dataset/training combination: "
        f"dataset_mode={dataset_mode}, training_mode={training_mode}"
    )


def load_sim_ids(path: str, sim_max_exclusive: Optional[int]) -> np.ndarray:
    """
    Loads simulation IDs and optionally keeps only sims below a cutoff.
    """

    sims = np.load(path)

    if sim_max_exclusive is not None:
        sims = sims[sims < sim_max_exclusive]

    return sims


class ChannelSelectDataset(torch.utils.data.Dataset):
    """
    Safety wrapper that ensures the dataset only returns requested channels.

    This is useful because some dataset classes may still return all 3 channels
    even when the model only expects KP.
    """

    def __init__(self, base_dataset, channels: str = "KP"):
        self.base_dataset = base_dataset
        self.channels = channels

    def _idx(self):
        if self.channels == "all":
            return [0, 1, 2]
        if self.channels == "KP":
            return [0, 1]
        if self.channels == "K":
            return [0]
        if self.channels == "P":
            return [1]
        if self.channels == "phi":
            return [2]

        raise ValueError(f"Bad channels: {self.channels}")

    def __len__(self):
        return len(self.base_dataset)

    def __getattr__(self, name):
        # Allows access to base dataset properties like sims/types/num_steps.
        return getattr(self.base_dataset, name)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        chans = self._idx()

        if len(item) == 3:
            feat, label, mask = item
            return feat[chans], label[chans], mask

        feat, label = item
        return feat[chans], label[chans]


def make_loaders(config: ExperimentConfig) -> Tuple[DataLoader, DataLoader]:
    """
    Builds train and validation DataLoaders for one experiment.
    """

    train_sims = load_sim_ids(config.train_sims_path, config.sim_max_exclusive)
    val_sims = load_sim_ids(config.val_sims_path, config.sim_max_exclusive)

    dataset_cls = get_dataset_class(config.dataset_mode, config.training_mode)

    kwargs = dict(config.dataset_kwargs or {})
    kwargs["channels"] = config.channels

    train_data = dataset_cls(train_sims, **kwargs)
    val_data = dataset_cls(val_sims, **kwargs)

    train_data = ChannelSelectDataset(train_data, config.channels)
    val_data = ChannelSelectDataset(val_data, config.channels)

    train_loader = DataLoader(
        train_data,
        batch_size=config.batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_data,
        batch_size=config.batch_size,
        shuffle=False,
    )

    return train_loader, val_loader